In [1]:
import sys
import os

# Ensure src/ is on the path so `lasr` is importable.
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from lasr.config import ModelConfig, InferenceConfig, PromptStyle
from lasr.data import load_esnli, build_few_shot_examples, build_prompts
from lasr.inference import load_model, generate_predictions
from lasr.metrics import run_evaluation

/workspace/LASR/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuration
All tunables in one place.

In [2]:
model_config = ModelConfig(model_name="google/gemma-3-1b-it")
inference_config = InferenceConfig(batch_size=8, max_new_tokens=256, downsample_rate=10)
prompt_style = PromptStyle.ONE_WORD
use_few_shot = False

ESNLI_URL = "https://raw.githubusercontent.com/OanaMariaCamburu/e-SNLI/refs/heads/master/dataset/esnli_dev.csv"

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {inference_config.batch_size}")
print(f"Downsample: 1/{inference_config.downsample_rate}")
print(f"Prompt:     {prompt_style.value}")
print(f"Few-shot:   {use_few_shot}")

Model:      google/gemma-3-1b-it
Device:     cuda
Batch size: 8
Downsample: 1/10
Prompt:     one_word
Few-shot:   False


# Setup
## Retrieve HF token
This will be different based on whether we run Colab vs locally.

In [3]:
import sys
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data Setup
## Load e-SNLI

In [4]:
esnli_df = load_esnli(ESNLI_URL)
esnli_df.head()

Downloading...


Done! 9842 rows loaded.


,pairID,gold_label,Sentence1,Sentence2,Explanation_1,Sentence1_marked_1,Sentence2_marked_1,Sentence1_Highlighted_1,Sentence2_Highlighted_1,Explanation_2,Sentence1_marked_2,Sentence2_marked_2,Sentence1_Highlighted_2,Sentence2_Highlighted_2,Explanation_3,Sentence1_marked_3,Sentence2_marked_3,Sentence1_Highlighted_3,Sentence2_Highlighted_3
0,4705552913.jpg#2r1n,neutral,Two women are embracing while holding to go pa...,The sisters are hugging goodbye while holding ...,The to go packages may not be from lunch.,Two women are embracing while holding to go pa...,The sisters are hugging goodbye while holding...,{},13,"Just because two women are embracing, does not...",Two women are embracing while holding to go pa...,The *sisters* are *hugging* *goodbye* while h...,{},"1,3,4",Two women do not have to be sisters. Embracin...,Two women are embracing while holding to go pa...,*The* *sisters* are *hugging* *goodbye* while...,{},"1,0,3,4,10,11,13,12"
1,4705552913.jpg#2r1e,entailment,Two women are embracing while holding to go pa...,Two woman are holding packages.,Saying the two women are holding packages is a...,Two women are embracing while holding *to* *g...,Two woman are *holding* *packages.*,"6,7,8","3,4",Sentence 1 states that two women are holding t...,*Two* *women* are embracing while *holding* t...,Two woman are holding packages.,"0,1,5,8",{},Women can embrace while they are holding packa...,Two *women* are *embracing* while holding to ...,Two woman are *holding* *packages.*,"1,3","3,4"
2,4705552913.jpg#2r1c,contradiction,Two women are embracing while holding to go pa...,The men are fighting outside a deli.,In the first sentence there is an action of af...,Two *women* are *embracing* while holding to g...,The *men* are *fighting* outside a deli.,"1,3","1,3",Women are different than men and embracing is ...,*Two* *women* are *embracing* while holding to...,The *men* are *fighting* outside a deli.,"0,1,3","1,3",First sentence features two women and the seco...,*Two* *women* are embracing while holding to g...,The *men* are fighting outside a deli.,"0,1",1
3,2407214681.jpg#0r1e,entailment,"Two young children in blue jerseys, one with t...",Two kids in numbered jerseys wash their hands.,Young children are kids. Jerseys with number 9...,"Two *young* *children* in blue *jerseys,* one...",Two *kids* in *numbered* *jerseys* wash their...,"2,1,10,9,15,16,5","1,3,4",TWO YOUNG CHILDREN IN JERSEY WASHING THEIR HAN...,"*Two* *young* *children* in blue jerseys, one...",Two kids in numbered *jerseys* wash their *ha...,"2,26,1,0,31","4,7",Kids is a synonym for children.,"Two young *children* in blue jerseys, one wit...",Two *kids* in numbered jerseys wash their hands.,2,1
4,2407214681.jpg#0r1n,neutral,"Two young children in blue jerseys, one with t...",Two kids at a ballgame wash their hands.,Two kids in jerseys watching their hands are n...,"Two young children in blue jerseys, one with t...",Two kids *at* *a* *ballgame* wash their hands.,{},"2,3,4",Even it two children are wearing jerseys they ...,"Two young children in blue jerseys, one with t...",Two kids at a *ballgame* wash their hands.,{},4,Just because two children are in jerseys washi...,"Two young children in blue jerseys, one with t...",Two *kids* *at* *a* *ballgame* *wash* *their*...,{},"4,2,1,3,5,6,7"


## Build Prompts

In [5]:
few_shot_examples = build_few_shot_examples(esnli_df, prompt_style) if use_few_shot else None
esnli_df["prompt"] = build_prompts(esnli_df, prompt_style, few_shot=use_few_shot, few_shot_examples=few_shot_examples)

print(esnli_df["prompt"].iloc[0])

<start_of_turn>user Determine if statement B is an entailment, contradiction or neutral with respect to statement A. Answer with a single word: entailment, contradiction, or neutral.
A: Two women are embracing while holding to go packages.
B: The sisters are hugging goodbye while holding to go packages after just eating lunch.
<end_of_turn>model


# Load Model

In [6]:
model, tokenizer = load_model(model_config)

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 630.35it/s, Materializing param=model.norm.weight]                                


# Inference

## Run Batched Generation

In [7]:
valid_mask = esnli_df["prompt"].notna()
valid_indices = esnli_df.index[valid_mask][:: inference_config.downsample_rate]
sampled_mask = esnli_df.index.isin(valid_indices)

prompts = esnli_df.loc[sampled_mask, "prompt"].tolist()
print(f"Running on {len(prompts)} / {valid_mask.sum()} samples (1 in every {inference_config.downsample_rate})")

decoded_outputs = generate_predictions(
    prompts, model, tokenizer, inference_config, device=model_config.device
)

Running on 985 / 9842 samples (1 in every 10)


Generating: 100%|██████████| 124/124 [10:52<00:00,  5.26s/it]

Generated 985 outputs


In [8]:
import torch
print(torch.__version__)

2.10.0+cu128


# Evaluation

In [9]:
esnli_df, report = run_evaluation(
    esnli_df, decoded_outputs, sampled_mask, prompt_style,
    model_name=model_config.model_name, few_shot=use_few_shot,
)
print(report)

Report saved to experiment_results/model-metrics_gemma-3-1b-it_one_word_one-shot.txt
Evaluated 708 / 985 rows (277 failed to parse)
               precision    recall  f1-score   support

contradiction       0.74      0.17      0.28       247
   entailment       0.48      0.75      0.58       234
      neutral       0.31      0.38      0.34       227

     accuracy                           0.43       708
    macro avg       0.51      0.44      0.40       708
 weighted avg       0.51      0.43      0.40       708



# SAE Activations
Hook GemmaScope 2 SAE and inspect the activations.

In [12]:
from IPython.display import display, IFrame
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

import plotly.express as px
import torch
import torch.nn as nn

_SAE_REPO_ID = "google/gemma-scope-2-4b-it"
_RELEASE_NAME = "gemma-scope-2-4b-it-res"
_SAE_ID = "layer_22_width_65k_l0_medium"

device = model_config.device

In [13]:
# SAE Lens doesn't provide an implementation for Gemma-3-1b yet.
class JumpReLUSAE(nn.Module):
  def __init__(self, d_in, d_sae, affine_skip_connection=False):
    # Encoder.
    super().__init__()
    self.w_enc = nn.Parameter(torch.zeros(d_in, d_sae))
    self.b_enc = nn.Parameter(torch.zeros(d_sae))
    self.threshold = nn.Parameter(torch.zeros(d_sae))

    # Decoder.
    self.w_dec = nn.Parameter(torch.zeros(d_sae, d_in))
    self.b_dec = nn.Parameter(torch.zeros(d_in))

    if affine_skip_connection:
      self.affine_skip_connection = nn.Parameter(torch.zeros(d_in, d_in))
    else:
      self.affine_skip_connection = None

  def encode(self, input_acts):
    print(f'W_enc type: {self.w_enc.dtype}')
    print(f'b_enc type: {self.b_enc.dtype}')
    pre_acts = input_acts @ self.w_enc + self.b_enc
    mask = (pre_acts > self.threshold)
    acts = mask * torch.nn.functional.relu(pre_acts)
    return acts

  def decode(self, acts):
    return acts @ self.w_dec + self.b_dec

  def forward(self, x):
    acts = self.encode(x)
    recon = self.decode(acts)
    if self.affine_skip_connection is not None:
      return recon + x @ self.affine_skip_connection
    return recon

In [14]:
path_to_params = hf_hub_download(
    repo_id=_SAE_REPO_ID,
    filename=f'resid_post/{_SAE_ID}/params.safetensors'
)

params = load_file(path_to_params)
print(f"\nSAE parameters:")
for name, tensor in params.items():
    print(f"  {name}: {tensor.shape}")


SAE parameters:
  b_dec: torch.Size([2560])
  b_enc: torch.Size([65536])
  threshold: torch.Size([65536])
  w_dec: torch.Size([65536, 2560])
  w_enc: torch.Size([2560, 65536])


In [15]:
d_model, d_sae = params['w_enc'].shape
sae = JumpReLUSAE(d_model, d_sae)
sae.load_state_dict(params)
sae.cuda()

JumpReLUSAE()

In [16]:
def gather_activations(model, target_layer, inputs):
  """Returns the activations in the target_layer of the model using inputs.

    Args:
      model: The HF model to use.
      target_layer: int, index of target layer.
      inputs: torch.Tensor, input to the model.

    Returns:
      torch.Tensor, activations in the target layer.
  """
  cache = {}

  def hook_fn(module, input, output):
      # Hugging Face models often return tuples (hidden_states, attention_weights, etc.)
      # We usually want the first element which is the hidden states.
      if isinstance(output, tuple):
          activations = output[0]
      else: # If it's just a tensor
          activations = output

      # Detach to ensure we don't keep the computation graph
      cache['activations'] = activations.detach()

  # Register the hook
  handle = model.model.language_model.layers[target_layer].register_forward_hook(hook_fn)

  try:
      with torch.no_grad():
          model(inputs)
  finally:
      # Ensure the hook is removed
      handle.remove()

  return cache['activations']

In [17]:
# --- Quick single-prompt demo: gather activations for GENERATED tokens only ---

TARGET_LAYER = 22  # must match the SAE (layer_22_width_65k_l0_medium)
MAX_NEW_TOKENS = 256

# 1. Tokenize the prompt
sample_prompt = esnli_df['prompt'].iloc[0]
prompt_ids = tokenizer.encode(sample_prompt, return_tensors='pt', add_special_tokens=True).to(device)
prompt_len = prompt_ids.shape[1]
print(f'Prompt tokens: {prompt_len}')

# 2. Generate the model's response (returns prompt + generated tokens)
full_ids = model.generate(input_ids=prompt_ids, max_new_tokens=MAX_NEW_TOKENS)
gen_len = full_ids.shape[1] - prompt_len
print(f'Generated tokens: {gen_len}')
print(f'Full sequence tokens: {full_ids.shape[1]}')

# 3. Run a forward pass on the full sequence and hook layer 22
residual_acts = gather_activations(model, TARGET_LAYER, full_ids)
print(f'Residual activations shape (full): {residual_acts.shape}')  # (1, prompt+gen, d_model)

# 4. Slice to keep only the generated-token activations
gen_acts = residual_acts[:, prompt_len:, :]
print(f'Generated-only activations shape: {gen_acts.shape}')  # (1, gen_len, d_model)
print(f'Type: {gen_acts.dtype}')

# 5. Encode through the JumpReLU SAE to get sparse features
sae_feature_acts = sae.encode(gen_acts.to(torch.float32))
print(f'SAE feature activations shape: {sae_feature_acts.shape}')  # (1, gen_len, d_sae)

# 6. Inspect features at the last generated token
last_token_features = sae_feature_acts[0, -1, :]  # (d_sae,)
n_active = (last_token_features > 0).sum().item()
print(f'\nActive features at last generated token: {n_active} / {last_token_features.shape[0]}')

# 7. Show the top-k most activated features
TOP_K = 10
top_vals, top_idxs = last_token_features.topk(TOP_K)
print(f'\nTop {TOP_K} SAE features at last generated token:')
for rank, (idx, val) in enumerate(zip(top_idxs.tolist(), top_vals.tolist()), 1):
    print(f'  #{rank}  feature {idx:>5d}  activation = {val:.4f}')

# 8. Print the generated text for reference
print(f'\n--- Generated Response ---')
print(tokenizer.decode(full_ids[0, prompt_len:], skip_special_tokens=True))

Prompt tokens: 69
Generated tokens: 6
Full sequence tokens: 75


AttributeError: 'Gemma3TextModel' object has no attribute 'language_model'

In [ ]:
px.line(
    cache['blocks.22.hook_resid_post.hook_sae_acts_post'][0, -1, :].cpu().numpy(),
    title='Feature activations at the final token position',
    labels={'index': 'Featutre', 'value': 'Activation'},
).show()

html_template = 'https://www.neuronpedia.org/gemma-3-1b-it/22-gemmascope-2-res-65k/{}?embed=true&embedexplanation=true&embedplots=true&embedsteer=true&embedactivations=true&embedlink=true&embedtest=true'

vals, inds = torch.topk(
    cache['blocks.22.hook_resid_post.hook_sae_acts_post'][0, -1, :], 5
)

# Fetch concept labels from Neuronpedia
from lasr.neuronpedia import get_neuronpedia_labels

np_model_id = "gemma-3-1b-it"
np_sae_id = "22-gemmascope-2-res-65k"
labels = get_neuronpedia_labels(np_model_id, np_sae_id, inds.tolist())

for val, ind in zip(vals, inds):
    concept = labels.get(ind.item())
    concept_str = f"  | concept: {concept}" if concept else ""
    print(f'Feature {ind} fired {val:.2f}{concept_str}')
    html = html_template.format(ind)
    print('URL: ' + html)
    display(IFrame(html, width=1200, height=300))